In [5]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/javier.escobar/code/github/skforecast
0.22.0


Benchmark diseñado para comparar el rendimiento de dos estrategias de
extracción de series temporales individuales desde un DataFrame con
MultiIndex (series_id, datetime).

El conjunto de prueba contiene 1000 series, cada una con 1000 valores
horarios, para un total de 1.000.000 de filas.

Se comparan:
1) acceso iterativo mediante .loc por cada identificador de serie
2) particionado mediante groupby(level=0)

Además del tiempo de ejecución, se verifica de forma robusta que las
series resultantes preservan tanto la metadata de frecuencia (.freq)
como la regularidad temporal real mediante pd.infer_freq().

El objetivo es evaluar simultáneamente rendimiento, escalabilidad y
preservación de la estructura temporal.

In [6]:
import pandas as pd
import numpy as np
import timeit
import statistics

# ==========================================
# Configuración del dataset
# ==========================================
n_series = 1000
n_values = 1000
expected_freq = "h"
n_runs = 5

# ==========================================
# Crear dataset de prueba
# ==========================================
dates = pd.date_range(
    start="2024-01-01",
    periods=n_values,
    freq=expected_freq
)

assert dates.freq is not None, "dates debe tener frecuencia definida antes de crear el MultiIndex"

series_ids = [f"serie_{i}" for i in range(n_series)]

index = pd.MultiIndex.from_product(
    [series_ids, dates],
    names=["series_id", "datetime"]
)

df = pd.DataFrame(
    {
        "value": np.random.rand(n_series * n_values)
    },
    index=index
)

first_col = "value"

print(f"DataFrame creado: {len(df):,} filas")
print(f"Número de series: {n_series}")
print(f"Valores por serie: {n_values}")
print(f"dates.freq: {dates.freq}")
print()

# ==========================================
# Método 1: usando loc
# ==========================================
def method_loc():
    return {
        sid: df.loc[sid][first_col].rename(sid)
        for sid in df.index.get_level_values(0).unique()
    }

def method_loc_2():
    return {
        sid: df.loc[sid] 
        for sid in df.index.levels[0]
    }

# ==========================================
# Método 2: usando groupby
# ==========================================
def method_groupby():
    return {
        sid: group[first_col].droplevel(0).rename(sid)
        for sid, group in df.groupby(level=0, sort=True)
    }

# ==========================================
# Verificación robusta de frecuencia
# ==========================================
def check_frequency_robust(series_dict, expected_freq="h"):
    lost_metadata = []
    irregular = []

    for sid, s in series_dict.items():
        idx = s.index

        # Verifica si se pierde metadata freq
        if idx.freq is None:
            lost_metadata.append(sid)

        # Verifica regularidad real
        inferred = pd.infer_freq(idx)

        if inferred is None:
            irregular.append((sid, "No inferible"))
        elif inferred.lower() != expected_freq.lower():
            irregular.append((sid, inferred))

    return {
        "lost_metadata_count": len(lost_metadata),
        "irregular_count": len(irregular),
        "lost_metadata_examples": lost_metadata[:5],
        "irregular_examples": irregular[:5],
    }

# ==========================================
# Benchmark
# ==========================================
print("Ejecutando benchmark...")

loc_times = timeit.repeat(method_loc, repeat=n_runs, number=1)
loc_2_times = timeit.repeat(method_loc_2, repeat=n_runs, number=1)
grp_times = timeit.repeat(method_groupby, repeat=n_runs, number=1)

loc_avg = statistics.mean(loc_times)
loc_2_avg = statistics.mean(loc_2_times)
grp_avg = statistics.mean(grp_times)

print("\n=== RESULTADOS DE VELOCIDAD ===")
print(f"LOC       -> promedio: {loc_avg:.4f} s")
print(f"LOC_2     -> promedio: {loc_2_avg:.4f} s")
print(f"GROUPBY   -> promedio: {grp_avg:.4f} s")
print(f"Speedup LOC   vs GROUPBY -> {loc_avg / grp_avg:.2f}x")
print(f"Speedup LOC_2 vs GROUPBY -> {loc_2_avg / grp_avg:.2f}x")
print()

# ==========================================
# Verificación de frecuencia
# ==========================================
print("Verificando frecuencia...")

loc_result = method_loc()
loc_2_result = method_loc_2()
grp_result = method_groupby()

loc_check = check_frequency_robust(loc_result, expected_freq)
loc_2_check = check_frequency_robust(loc_2_result, expected_freq)
grp_check = check_frequency_robust(grp_result, expected_freq)

print("\n=== VERIFICACIÓN ROBUSTA DE FRECUENCIA ===")

print("\nMétodo LOC")
print(f"Series con metadata freq perdida: {loc_check['lost_metadata_count']}")
print(f"Series irregulares reales: {loc_check['irregular_count']}")
print(f"Ejemplos metadata perdida: {loc_check['lost_metadata_examples']}")
print(f"Ejemplos irregulares: {loc_check['irregular_examples']}")

print("\nMétodo LOC_2")
print(f"Series con metadata freq perdida: {loc_2_check['lost_metadata_count']}")
print(f"Series irregulares reales: {loc_2_check['irregular_count']}")
print(f"Ejemplos metadata perdida: {loc_2_check['lost_metadata_examples']}")
print(f"Ejemplos irregulares: {loc_2_check['irregular_examples']}")

print("\nMétodo GROUPBY")
print(f"Series con metadata freq perdida: {grp_check['lost_metadata_count']}")
print(f"Series irregulares reales: {grp_check['irregular_count']}")
print(f"Ejemplos metadata perdida: {grp_check['lost_metadata_examples']}")
print(f"Ejemplos irregulares: {grp_check['irregular_examples']}")

DataFrame creado: 1,000,000 filas
Número de series: 1000
Valores por serie: 1000
dates.freq: <Hour>

Ejecutando benchmark...

=== RESULTADOS DE VELOCIDAD ===
LOC       -> promedio: 0.4973 s
LOC_2     -> promedio: 0.4352 s
GROUPBY   -> promedio: 0.0767 s
Speedup LOC   vs GROUPBY -> 6.48x
Speedup LOC_2 vs GROUPBY -> 5.67x

Verificando frecuencia...

=== VERIFICACIÓN ROBUSTA DE FRECUENCIA ===

Método LOC
Series con metadata freq perdida: 0
Series irregulares reales: 0
Ejemplos metadata perdida: []
Ejemplos irregulares: []

Método LOC_2
Series con metadata freq perdida: 0
Series irregulares reales: 0
Ejemplos metadata perdida: []
Ejemplos irregulares: []

Método GROUPBY
Series con metadata freq perdida: 0
Series irregulares reales: 0
Ejemplos metadata perdida: []
Ejemplos irregulares: []


In [7]:
# ==========================================
# Verificación de equivalencia de resultados
# ==========================================
print("Verificando equivalencia entre métodos...")

for sid in loc_result:
    # LOC vs GROUPBY (ambos devuelven Series)
    pd.testing.assert_series_equal(loc_result[sid], grp_result[sid])

    # LOC_2 devuelve DataFrame, comparar con LOC tras extraer la columna
    pd.testing.assert_series_equal(
        loc_2_result[sid][first_col].rename(sid),
        loc_result[sid]
    )

print("OK: Los tres métodos devuelven resultados equivalentes.")

# ==========================================
# Benchmark con series de longitud irregular
# ==========================================
print("\n" + "=" * 50)
print("BENCHMARK CON SERIES DE LONGITUD IRREGULAR")
print("=" * 50)

np.random.seed(42)
n_series_irr = 200
min_len = 100
max_len = 1000

frames = []
for i in range(n_series_irr):
    length = np.random.randint(min_len, max_len + 1)
    dates_i = pd.date_range(start="2024-01-01", periods=length, freq=expected_freq)
    idx = pd.MultiIndex.from_arrays(
        [[f"serie_{i}"] * length, dates_i],
        names=["series_id", "datetime"]
    )
    frames.append(pd.DataFrame({"value": np.random.rand(length)}, index=idx))

df_irr = pd.concat(frames)

print(f"DataFrame irregular creado: {len(df_irr):,} filas, {n_series_irr} series")
print(f"Longitudes: min={min_len}, max={max_len}")

def method_loc_irr():
    return {
        sid: df_irr.loc[sid]["value"].rename(sid)
        for sid in df_irr.index.get_level_values(0).unique()
    }

def method_loc_2_irr():
    return {
        sid: df_irr.loc[sid]
        for sid in df_irr.index.levels[0]
    }

def method_groupby_irr():
    return {
        sid: group["value"].droplevel(0).rename(sid)
        for sid, group in df_irr.groupby(level=0, sort=True)
    }

# Benchmark
loc_irr_times = timeit.repeat(method_loc_irr, repeat=n_runs, number=1)
loc_2_irr_times = timeit.repeat(method_loc_2_irr, repeat=n_runs, number=1)
grp_irr_times = timeit.repeat(method_groupby_irr, repeat=n_runs, number=1)

print(f"\nLOC       -> promedio: {statistics.mean(loc_irr_times):.4f} s")
print(f"LOC_2     -> promedio: {statistics.mean(loc_2_irr_times):.4f} s")
print(f"GROUPBY   -> promedio: {statistics.mean(grp_irr_times):.4f} s")
print(f"Speedup LOC   vs GROUPBY -> {statistics.mean(loc_irr_times) / statistics.mean(grp_irr_times):.2f}x")
print(f"Speedup LOC_2 vs GROUPBY -> {statistics.mean(loc_2_irr_times) / statistics.mean(grp_irr_times):.2f}x")

# Equivalencia con longitudes irregulares
loc_irr_result = method_loc_irr()
loc_2_irr_result = method_loc_2_irr()
grp_irr_result = method_groupby_irr()

for sid in loc_irr_result:
    pd.testing.assert_series_equal(loc_irr_result[sid], grp_irr_result[sid])
    pd.testing.assert_series_equal(
        loc_2_irr_result[sid]["value"].rename(sid),
        loc_irr_result[sid]
    )

print("\nOK: Equivalencia verificada también con series de longitud irregular.")

# Verificación de frecuencia con series irregulares
loc_irr_check = check_frequency_robust(loc_irr_result, expected_freq)
grp_irr_check = check_frequency_robust(grp_irr_result, expected_freq)

print(f"\nFrecuencia perdida (LOC):     {loc_irr_check['lost_metadata_count']} / {n_series_irr}")
print(f"Frecuencia perdida (GROUPBY): {grp_irr_check['lost_metadata_count']} / {n_series_irr}")
print(f"Series irregulares (LOC):     {loc_irr_check['irregular_count']}")
print(f"Series irregulares (GROUPBY): {grp_irr_check['irregular_count']}")

Verificando equivalencia entre métodos...
OK: Los tres métodos devuelven resultados equivalentes.

BENCHMARK CON SERIES DE LONGITUD IRREGULAR
DataFrame irregular creado: 110,705 filas, 200 series
Longitudes: min=100, max=1000

LOC       -> promedio: 0.0281 s
LOC_2     -> promedio: 0.0184 s
GROUPBY   -> promedio: 0.0119 s
Speedup LOC   vs GROUPBY -> 2.36x
Speedup LOC_2 vs GROUPBY -> 1.55x

OK: Equivalencia verificada también con series de longitud irregular.

Frecuencia perdida (LOC):     0 / 200
Frecuencia perdida (GROUPBY): 0 / 200
Series irregulares (LOC):     0
Series irregulares (GROUPBY): 0
